In [ ]:
import sys
sys.path.append("..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import FEATURES, PROCESSED_DIR, RANDOM_STATE
from src.extraction import load_raw_data
from src.preprocessing import (make_clean, prepare_clustering, run_full_pca,
                               n_components_for_threshold)
from src.clustering import (grid_kmeans, fit_kmeans)

## Feature Story 1 : EDA

In [ ]:
df = load_raw_data()
print(df.shape)          # dimensions
print(df.dtypes)         # types
df.head()

### # Valeurs manquantes (nombre + %), triées par ordre décroissant

In [ ]:
manquants = df.isna().sum().to_frame("nb_manquants")
manquants["pct"] = (manquants["nb_manquants"] / len(df) * 100).round(2)
manquants = manquants.sort_values("nb_manquants", ascending=False)
display(manquants)

print("Doublons :", df.duplicated().sum())
df[FEATURES].describe()

### # Histogrammes + KDE, skewness dans le titre

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()
for ax, col in zip(axes, FEATURES):
    sns.histplot(df[col], kde=True, ax=ax)
    ax.set_title(f"{col}\nskew = {df[col].skew():.2f}")
for ax in axes[len(FEATURES):]:
    ax.axis("off")                      # cache la case vide (7 variables / 8 cases)
plt.tight_layout(); plt.show()

### Heatmap des corrélations

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(df[FEATURES].corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Matrice de corrélation"); plt.show()

### Outliers

In [ ]:
# Boxplots (outliers)
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()
for ax, col in zip(axes, FEATURES):
    sns.boxplot(y=df[col], ax=ax)
    ax.set_title(col)
for ax in axes[len(FEATURES):]:
    ax.axis("off")
plt.tight_layout(); plt.show()

# Nombre d'outliers
rows = []
for col in FEATURES:
    q1, q3 = df[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    n = ((df[col] < q1 - 1.5 * iqr) | (df[col] > q3 + 1.5 * iqr)).sum()
    rows.append({"variable": col, "nb_outliers": n, "pct": round(n / len(df) * 100, 1)})
pd.DataFrame(rows)

## Feature Story 2 : Prétraitement

### Nettoyage minimal + sauvegarde AVANT toute transformation 

In [ ]:
df_clean = make_clean(df)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
df_clean.to_csv(PROCESSED_DIR / "df_clean.csv", index=False)
print(df_clean.shape)

### Branche clustering

In [ ]:
df_log, df_prepare_clustering = prepare_clustering(df_clean)
df_prepare_clustering.to_csv(PROCESSED_DIR / "df_prepare_clustering.csv", index=False)

# Histogrammes après log
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()
for ax, col in zip(axes, FEATURES):
    sns.histplot(df_log[col], kde=True, ax=ax)
    ax.set_title(f"{col} (log)\nskew = {df_log[col].skew():.2f}")
for ax in axes[len(FEATURES):]:
    ax.axis("off")
plt.tight_layout(); plt.show()

# Tableau avant / après
skew_table = pd.DataFrame({
    "skew_avant": df_clean[FEATURES].skew(),
    "skew_apres_log": df_log.skew(),
}).round(2)
skew_table

### Une vision provisoire

In [ ]:
skew_table["variation_absolue"] = (
    skew_table["skew_apres_log"].abs()
    - skew_table["skew_avant"].abs()
)

skew_table

###  PCA complète 

In [ ]:
pca, X_pca = run_full_pca(df_prepare_clustering)
var = pca.explained_variance_ratio_
cumul = np.cumsum(var)
n_80 = n_components_for_threshold(pca, 0.80)

display(pd.DataFrame({"composante": range(1, len(var) + 1),
                      "variance": var.round(3), "variance_cumulee": cumul.round(3)}))

plt.figure(figsize=(7, 4))
plt.plot(range(1, len(cumul) + 1), cumul, marker="o")
plt.axhline(0.80, color="red", linestyle="--", label="seuil 80 %")
plt.axvline(n_80, color="green", linestyle="--", label=f"{n_80} composantes")
plt.xlabel("Nombre de composantes"); plt.ylabel("Variance cumulée")
plt.title("Variance expliquée cumulée (PCA)"); plt.legend(); plt.show()
print("Première estimation :", n_80, "composantes")

## Features story 3

### Grille K-means

In [ ]:
res_km = grid_kmeans(X_pca, comps=range(2, 6), ks=range(2, 8))

# Heatmap composantes x k (valeur = silhouette)
pivot = res_km.pivot(index="n_components", columns="k", values="silhouette")
plt.figure(figsize=(8, 4))
sns.heatmap(pivot, annot=True, fmt=".3f", cmap="viridis")
plt.title("Silhouette : composantes PCA x k"); plt.show()

# Courbes elbow et silhouette (vs k), une ligne par nombre de composantes
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for n, g in res_km.groupby("n_components"):
    axes[0].plot(g["k"], g["inertia"], marker="o", label=f"{n} comp.")
    axes[1].plot(g["k"], g["silhouette"], marker="o", label=f"{n} comp.")
axes[0].set_title("Elbow (inertie)"); axes[1].set_title("Silhouette")
for ax in axes: ax.set_xlabel("k"); ax.legend()
plt.show()


res_km.sort_values("silhouette", ascending=False).head(10)

### Choix de la combinaison

In [ ]:
ok = res_km[res_km["min_pct"] >= 10].sort_values("silhouette", ascending=False)
display(ok.head())

BEST_N = int(ok.iloc[0]["n_components"])
BEST_K = int(ok.iloc[0]["k"])
print("Choix :", BEST_N, "composantes, k =", BEST_K)

km, X_km = fit_kmeans(X_pca, BEST_N, BEST_K)
df_clean["cluster_kmeans"] = km.labels_
df_clean["cluster_kmeans"].value_counts(normalize=True).round(3)